# Terraform Code Generation & Validation Agent

Ce notebook utilise un agent LangChain pour générer et valider du code Terraform.

**Fonctionnalités:**
- 🤖 Génération automatique de code Terraform basée sur des spécifications
- 📚 Base de connaissances (ChromaDB) avec les best practices Terraform
- ✅ Validation automatique (terraform validate, terraform plan)
- 🔍 Revue de code avec suggestions de corrections
- 🎯 Routage intelligent des modèles (Claude + Ollama)

**Workflow:**
1. Configuration de l'agent et chargement des best practices
2. Chargement d'un prompt utilisateur depuis `user_prompts/`
3. Génération du code Terraform dans `work_dir` (configuré dans le prompt utilisateur)
4. Validation et revue automatiques
5. Corrections itératives si nécessaire

**Modèles utilisés:**
- Agent principal: Claude Haiku 4.5
- Revue de code: Qwen2.5-coder (Ollama)
- Embeddings: nomic-embed-text (Ollama)

## Étape 1: Imports et Configuration de Base

Cette section charge tous les modules nécessaires:
- **dotenv**: Pour charger les variables d'environnement (clés API, endpoints)
- **terraform_agent**: Classes principales pour la génération et validation de code

Le projet est automatiquement ajouté au `sys.path` pour permettre l'import des modules locaux.

In [1]:
# ============================================================================
# TERRAFORM CODE GENERATION & VALIDATION AGENT
# ============================================================================

from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Import agent classes
import sys
sys.path.insert(0, str(Path.cwd().parent))

from terraform_agent import Config, PromptManager, KnowledgeBase, TerraformGenerator

print("✓ All imports loaded successfully")

✓ All imports loaded successfully


## Étape 2: Configuration du Logging

Configuration du système de logs pour suivre l'exécution de l'agent:
- **INFO level**: Affiche les étapes principales (chargement docs, appels API, validation)
- **DEBUG level**: Pour le module `terraform_agent` - détails complets de l'exécution

Les logs permettent de comprendre les décisions de l'agent et de débugger les problèmes.

In [2]:
# ============================================================================
# CONFIGURE LOGGING
# ============================================================================

import logging

# Configure logging to see all DEBUG and above messages
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)

# Optional: Set specific loggers to DEBUG for more detail
logging.getLogger('terraform_agent').setLevel(logging.DEBUG)

print("✓ Logging configured - you will now see INFO level logs and above")

✓ Logging configured - you will now see INFO level logs and above


## Étape 3: Initialisation des Composants

Cette étape configure et initialise tous les composants nécessaires:

### 3.1 Configuration (`Config`)
- Définit les chemins (project root, work dir, rules, prompts)
- Configure les modèles (Claude pour l'agent, Ollama pour la revue)
- Active le routage intelligent des modèles pour optimiser les coûts

### 3.2 Prompts (`PromptManager`)
- Charge les prompts système et utilisateur depuis `prompts/`
- Gère les templates pour la génération, validation et revue

### 3.3 Base de Connaissances (`KnowledgeBase`)
- Charge les best practices Terraform depuis `rules/`
- Indexe les documents dans ChromaDB avec embeddings Ollama
- Permet la recherche sémantique pendant la génération

### 3.4 Agent (`TerraformAgent`)
- Initialise l'agent LangChain avec Claude
- Configure les outils disponibles (init, validate, plan, review)
- Active le routage de modèles pour certaines tâches (Ollama pour parsing/review, Claude pour raisonnement)

In [3]:
# ============================================================================
# INITIALIZE COMPONENTS
# ============================================================================

# Get project root (parent of notebooks directory)
project_root = Path.cwd().parent

# Initialize configuration
config = Config(base_dir=project_root)
print(f"Project Root: {config.PROJECT_ROOT}")
print(f"Work Dir: {config.WORK_DIR}")
print(f"Docs Dir: {config.RULES_DIR}")

# Initialize prompt manager
prompts = PromptManager(config)
print("\n✓ Prompts loaded")

# Initialize knowledge base (loads and indexes docs)
knowledge_base = KnowledgeBase(config)

# Initialize agent
agent = TerraformGenerator(config, prompts, knowledge_base)

Project Root: /Users/melkouhen/audit-tools/test-langchain
Work Dir: /Users/melkouhen/audit-tools/test-langchain/work
Docs Dir: /Users/melkouhen/audit-tools/test-langchain/rules

✓ Prompts loaded
📚 Loading knowledge base...
  ✓ Loaded 29 document(s) from /Users/melkouhen/audit-tools/test-langchain/rules
  ✓ Split into 232 chunks
  Creating new vectorstore database...
  Indexing 232 documents...


14:54:53 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
14:54:53 - terraform_agent.tools - INFO - Initializing tools with config, prompts, and knowledge base
14:54:53 - terraform_agent.tools - INFO - Tools initialized - Review model: qwen2.5-coder:7b-instruct
14:54:53 - terraform_agent.tools - INFO - Model router enabled - Ollama tasks: {'review', 'summarization', 'parsing'}


  ✓ Vectorstore created and indexed
🤖 Setting up agent...
  ✓ System prompt loaded (12491 chars)
  ✓ User prompt loaded (1041 chars)
  ✓ Agent created with tools:
    - load_module_spec (module specifications)
    - search_knowledge_base (best practices)
    - terraform_init (initialize working directory)
    - terraform_validate (validate configuration)
    - terraform_plan (preview changes)
    - review_and_fix_code (code review)
  ✓ Model routing:
    - Ollama tasks: review, summarization, parsing
      • review: qwen2.5-coder:14b
      • summarization: qwen3.5:9b
      • parsing: qwen2.5-coder:7b-instruct


## Étape 4: Exécution de l'Agent

Cette cellule exécute le workflow complet de génération:

### Input
- Charge un prompt utilisateur depuis `user_prompts/` (ex: spécifications d'infrastructure)
- Le prompt décrit les ressources Terraform à créer

### Workflow de l'Agent
1. **Analyse** du prompt utilisateur et des spécifications
2. **Recherche** des best practices dans la base de connaissances
3. **Génération** du code Terraform dans `work/dev/`
4. **Validation** avec `terraform init` et `terraform validate`
5. **Plan** avec `terraform plan` pour vérifier les changements
6. **Revue** automatique du code avec détection des problèmes
7. **Corrections** itératives si nécessaire

### Output
- Code Terraform généré et validé dans `work/dev/`
- Rapport de validation et revue
- Logs détaillés de chaque étape

### Phase Tracking avec Callbacks
Le notebook utilise maintenant `DetailedTerraformCallback` pour afficher en temps réel:
- 📋 **PLANNING**: Analyse des spécifications et recherche de best practices
- 🔧 **GENERATION**: Création du code Terraform
- ✅ **VALIDATION**: terraform init, validate, plan
- 🔍 **CODE_REVIEW**: Revue automatique et corrections
- **Tool calls**: Affichage de chaque appel d'outil avec statut (→ Calling, ✅/❌ completed)
- **Execution summary**: Durées par phase, security checks, best practices appliquées

**Note:** L'agent utilise le routage de modèles pour optimiser les coûts:
- Claude Haiku 4.5 pour le raisonnement et la génération
- Ollama (Qwen) pour la revue de code et le parsing

In [ ]:
# ============================================================================
# EXECUTE AGENT
# ============================================================================

# Import callback for phase tracking
from terraform_agent.callbacks import DetailedTerraformCallback

# Create callback instance
callback = DetailedTerraformCallback(verbose=True)

# Load user prompt from file
prompt_filename = "user_prompts/2-cloudrun.md"  # Change filename as needed
user_prompt = (Path.cwd().parent / prompt_filename).read_text()

config.WORK_DIR = Path("/Users/melkouhen/audit-tools/test-langchain/work-02")

# Execute agent with callbacks for real-time phase tracking
result = agent.run(user_prompt=user_prompt, callbacks=[callback], clean_work_dir=True)
print(f"\n🎯 Agent Output:\n{result}")

14:54:53 - terraform_agent.callbacks - INFO - Phase started: GENERATION (triggered by llm)
14:54:53 - terraform_agent.callbacks - DEBUG - LLM generation started


🛠️  Preparing workspace...
  🧹 Cleaning work directory: /Users/melkouhen/audit-tools/test-langchain/work-02

🚀 Starting Terraform Code Generation Agent
Timestamp: 2026-05-13 14:54:53

📝 Agent is running...
    (Agent will autonomously call: terraform_init, terraform_validate, terraform_plan, review_and_fix_code)
--------------------------------------------------------------------------------

🔧 PHASE: GENERATION


14:54:56 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
14:54:56 - terraform_agent.callbacks - DEBUG - LLM generation completed
14:54:56 - terraform_agent.callbacks - DEBUG - Tool started: write_todos
14:54:56 - terraform_agent.callbacks - DEBUG - Tool ended: write_todos
14:54:56 - terraform_agent.callbacks - DEBUG - LLM generation started


   → Calling write_todos
   ✅ write_todos completed


14:55:00 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
14:55:00 - terraform_agent.callbacks - DEBUG - LLM generation completed
14:55:00 - terraform_agent.callbacks - INFO - Phase ended: GENERATION (6.77s)
14:55:00 - terraform_agent.callbacks - INFO - Phase started: PLANNING (triggered by search_knowledge_base)
14:55:00 - terraform_agent.callbacks - INFO - Phase ended: GENERATION (6.77s)
14:55:00 - terraform_agent.callbacks - DEBUG - Tool started: search_knowledge_base
14:55:00 - terraform_agent.callbacks - DEBUG - Tool started: search_knowledge_base
14:55:00 - terraform_agent.callbacks - DEBUG - Tool started: search_knowledge_base
14:55:00 - terraform_agent.callbacks - DEBUG - Tool started: search_knowledge_base
14:55:00 - terraform_agent.callbacks - INFO - Phase started: PLANNING (triggered by search_knowledge_base)
14:55:00 - terraform_agent.callbacks - DEBUG - Tool started: search_knowledge_base
14:55:00 - terraform_agent.callbacks - DEB

...

   → Calling read_file
   ✅ read_file completed


15:00:16 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
15:00:16 - terraform_agent.callbacks - DEBUG - LLM generation completed
15:00:16 - terraform_agent.callbacks - DEBUG - Tool started: write_file
15:00:16 - terraform_agent.callbacks - DEBUG - Tool ended: write_file
15:00:16 - terraform_agent.callbacks - DEBUG - LLM generation started


   → Calling write_file
   ✅ write_file completed


15:00:46 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
15:00:46 - terraform_agent.callbacks - DEBUG - LLM generation completed
15:00:46 - terraform_agent.callbacks - DEBUG - Tool started: write_file
15:00:46 - terraform_agent.callbacks - DEBUG - Tool ended: write_file
15:00:46 - terraform_agent.callbacks - DEBUG - LLM generation started


   → Calling write_file
   ✅ write_file completed
